# Week 01 · Crawl Prime Lands (Depth 3) → Markdown + JSONL

**Objective**: Crawl https://www.primelands.lk to max depth 3, save as Markdown files and consolidated JSONL corpus.

**Architecture**: Uses `PrimeLandsWebCrawler` service from `application.ingest_document_service`

**Provider Support**: Supports OpenRouter (multi-provider) or direct OpenAI API via `.env` configuration

In [1]:
# Cell 2: Imports & Environment Setup
import os
import sys
import json
import time
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
import nest_asyncio

# Enable nested asyncio
nest_asyncio.apply()

# Fix for Windows asyncio subprocess issues with Playwright
import asyncio
if hasattr(asyncio, 'WindowsSelectorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: OpenAI
📁 Project root: e:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands


In [2]:
# Cell 3: Load Configuration
from config import (
    validate, dump, CRAWL_OUT_DIR, MARKDOWN_DIR
)

# Validate and display config
try:
    validate()
    dump()
except Exception as e:
    print(f"⚠️  Config note: {e}")

# Ensure directories exist
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Output directories ready:")
print(f"   - Markdown: {MARKDOWN_DIR}")
print(f"   - JSONL: {CRAWL_OUT_DIR}")


CONFIGURATION (NON-SECRETS ONLY)

🌐 Provider:
   Provider: openai
   Model Tier: general
   Chat Model: gpt-4o-mini
   Embedding Model: text-embedding-3-large

📁 Directories:
   Data Root: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data
   Vector Store: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\vectorstore
   Markdown: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\primelands_markdown
   Cache: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\cag_cache

🔧 Chunking:
   Fixed Size: 800 tokens
   Fixed Overlap: 100 tokens
   Sliding Window: 512 tokens
   Sliding Stride: 256 tokens
   Parent-Child: 250 → 1200 tokens
   Late Chunk: 1000 → 300 tokens

🔍 Retrieval:
   Top-K Results: 4
  

## Import Crawler Service

Using `PrimeLandsWebCrawler` from application layer (NOT defined here!)

In [3]:
# Cell 4: Import Web Crawler Service
from application.ingest_document_service import PrimeLandsWebCrawler

print("✅ PrimeLandsWebCrawler loaded from service layer")
print("📍 Location: application.ingest_document_service.web_crawler")

✅ PrimeLandsWebCrawler loaded from service layer
📍 Location: application.ingest_document_service.web_crawler


## Crawl Configuration

In [4]:
# Cell 5: Crawl Configuration
BASE_URL = "https://www.primelands.lk"

START_PATHS = [
    "/", "/contact-us", "/news", "/land", "/house", "/apartment/complete", "/apartment/ongoing/",
    "/upcoming-projects/", "/sell-your-land/", "/services/", "/testimonial/", "/virtual-tour/",
    "/kyc/", "/about-us/"
]

START_URLS = [BASE_URL + path for path in START_PATHS]

EXCLUDE_PATTERNS = [
    "/login", "/terms", "/privacy", "/admin",
    "/images/", "/downloads/", "/media/"
]

MAX_DEPTH = 3
REQUEST_DELAY = 2.0
JSONL_PATH = CRAWL_OUT_DIR / "primelands_docs.jsonl"

print(f"🌐 Crawl config:")
print(f"   - Start URLs: {len(START_URLS)}")
print(f"   - Max depth: {MAX_DEPTH}")
print(f"   - Request delay: {REQUEST_DELAY}s")

🌐 Crawl config:
   - Start URLs: 14
   - Max depth: 3
   - Request delay: 2.0s


## Execute Crawl

In [5]:
# Cell 6: Run Crawl with Service
start_time = time.time()

# Initialize crawler service
crawler = PrimeLandsWebCrawler(
    base_url=BASE_URL,
    max_depth=MAX_DEPTH,
    exclude_patterns=EXCLUDE_PATTERNS
)

print(f"\n🚀 Starting crawl at {time.strftime('%H:%M:%S')}\n")
documents = crawler.crawl(START_URLS, request_delay=REQUEST_DELAY)

elapsed = time.time() - start_time
print(f"\n✅ Crawl complete in {elapsed:.1f}s")
print(f"📄 Documents collected: {len(documents)}")
print(f"🔗 URLs visited: {len(crawler.visited)}")


🚀 Starting crawl at 16:37:36

🔍 [0] https://www.primelands.lk/
   ✅ Saved (554 chars, 45 links found)
   📎 Added 45 new URLs to queue (depth 1)
   📊 Progress: 1 docs saved, 1 visited, 58 in queue
🔍 [0] https://www.primelands.lk/contact-us
   ✅ Saved (124 chars, 0 links found)
   📊 Progress: 2 docs saved, 2 visited, 57 in queue
🔍 [0] https://www.primelands.lk/news
   ✅ Saved (124 chars, 0 links found)
   📊 Progress: 3 docs saved, 3 visited, 56 in queue
🔍 [0] https://www.primelands.lk/land
   ✅ Saved (564 chars, 71 links found)
   📎 Added 57 new URLs to queue (depth 1)
   📊 Progress: 4 docs saved, 4 visited, 112 in queue
🔍 [0] https://www.primelands.lk/house
   ✅ Saved (566 chars, 35 links found)
   📎 Added 21 new URLs to queue (depth 1)
   📊 Progress: 5 docs saved, 5 visited, 132 in queue
🔍 [0] https://www.primelands.lk/apartment/complete
   ✅ Saved (124 chars, 0 links found)
   📊 Progress: 6 docs saved, 6 visited, 131 in queue
🔍 [0] https://www.primelands.lk/apartment/ongoing/
   ✅ Sa

In [7]:
# Cell 7: Save Outputs
# Save markdown files
import re
import hashlib

for i, doc in enumerate(documents):
    url_path = urlparse(doc['url']).path.strip('/').replace('/', '_')
    if not url_path:
        url_path = "homepage"

    safe_slug = re.sub(r'[^A-Za-z0-9_-]+', '-', url_path).strip('-')[:80]
    url_hash = hashlib.sha1(doc['url'].encode('utf-8')).hexdigest()[:10]
    filename = f"{i:03d}_{safe_slug}_{url_hash}.md"
    
    md_file = MARKDOWN_DIR / filename
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write(f"# {doc['title']}\n\n")
        f.write(f"**URL**: {doc['url']}\n\n")
        f.write(f"**Depth**: {doc['depth_level']}\n\n")
        f.write("---\n\n")
        f.write(doc['content'])

print(f"✅ Saved {len(documents)} markdown files to {MARKDOWN_DIR}")

# Save JSONL
with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"✅ Saved JSONL corpus to {JSONL_PATH}")

✅ Saved 140 markdown files to E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\primelands_markdown
✅ Saved JSONL corpus to E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\primelands_docs.jsonl


## Quality Checks

In [8]:
# Cell 8: Quality Checks
import random

print("🔍 Quality Checks:\n")

# Check markdown files
md_files = list(MARKDOWN_DIR.glob("*.md"))
print(f"1️⃣  Markdown files: {len(md_files)}")

if len(md_files) >= 20:
    print(f"   ✅ Good! Got {len(md_files)} pages")
elif len(md_files) >= 10:
    print(f"   ⚠️  Only {len(md_files)} pages (site may be small)")
else:
    raise AssertionError(f"❌ Too few pages: {len(md_files)}")

# Check JSONL
assert JSONL_PATH.exists(), f"❌ JSONL not found"
print(f"\n2️⃣  JSONL file: {JSONL_PATH.stat().st_size:,} bytes")

# Sample inspection
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    all_docs = [json.loads(line) for line in f]

samples = random.sample(all_docs, min(3, len(all_docs)))
print(f"\n3️⃣  Random samples:\n")

for i, doc in enumerate(samples, 1):
    print(f"   Sample {i}:")
    print(f"   - URL: {doc['url']}")
    print(f"   - Title: {doc['title']}")
    print(f"   - Words: {len(doc['content'].split())}")
    print()

print("✅ All quality checks passed!")

🔍 Quality Checks:

1️⃣  Markdown files: 159
   ✅ Good! Got 159 pages

2️⃣  JSONL file: 588,510 bytes

3️⃣  Random samples:

   Sample 1:
   - URL: https://www.primelands.lk/house/city/Gampaha/en
   - Title: House for Sale  in Gampaha | Houses by Prime Lands
   - Words: 14

   Sample 2:
   - URL: https://www.primelands.lk/portfolio-property/ANURADHAPURA-PALADIKULAMA/en
   - Title: ANURADHAPURA - PALADIKULAMA
   - Words: 14

   Sample 3:
   - URL: https://www.primelands.lk/house/sin
   - Title: Homes for Sale by Prime Lands | Designed for Modern Living
   - Words: 18

✅ All quality checks passed!
